<a href="https://colab.research.google.com/github/brainExplorer/image-classification-cifar10-cnn-transfer-learning/blob/main/Image_Classification_Project_CIFAR_10_to_CNN_and_Transfer_Learning_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##DataSet: CIFAR-10
The CIFAR-10 consists of 32 x 32 RGB images of 10 classes:

- 50,000 training, 10,000 test images

- Classes: airplane, car, bird, cat, deer, dog, frog, horse, ship, truck

 > <details> <summary>Hint</summary> num_workers ve pin_memory kullanımı veri yükleme hızını önemli ölçüde artırır.
 </details>

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms

# Training set With Data augmentation
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])


transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])


trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

# Let's divide the training set as 90% training, 10% verification
train_size = int(0.9 * len(trainset))
val_size = len(trainset) - train_size
trainset, valset = torch.utils.data.random_split(trainset, [train_size, val_size])

batch_size = 128
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
valloader = torch.utils.data.DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

100%|██████████| 170M/170M [2:20:01<00:00, 20.3kB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


##CNN From Scratch

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CNNFromScratch(nn.Module):
    def __init__(self, num_classes=10):
        super(CNNFromScratch, self).__init__()
        # Convolutional blocks
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)
        self.bn4 = nn.BatchNorm2d(256)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)

        # Fully connected layers
        self.fc1 = nn.Linear(256, num_classes)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = self.dropout(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        return x

# Architectural Summary:
| Layer | Output Size | Number of Parameters |
|-------|-------------:|---------------------:|
| Conv1 + BN + Pool | 16×16×32 | ~896 |
| Conv2 + BN + Pool | 8×8×64 | ~18,496 |
| Conv3 + BN + Pool | 4×4×128 | ~73,856 |
| Conv4 + BN + Pool | 2×2×256 | ~295,168 |
| GAP + FC | 10 | ~2,570 |
| **Total** | — | **~391,000** |

> **📌 Note:** The use of **Batch Normalization (BN)** accelerates the training process and improves training stability **[5, pp. 100–104]**.

## Transfer Learning: Fine-Tuning with ResNet-18

In [ ]:
import torchvision.models as models

def get_resnet18(num_classes=10, freeze_backbone=True):
  weights = models
  model = models.resnet18(weights=weights)

  # Stop the backbone(optional)
  if freeze_backbone:
    for param in model.parameters():
      param.requires_grad = False

  num_features = model.fc.in_features
  model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
  return model

 # To Fine-tuning (backbone is open)
model_resnet = get_resnet18(num_classes=10, freeze_backbone=False)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 168MB/s]


### Fine-Tuning Strategies

| Strategy | Description | Typical Use Case |
|----------|-------------|------------------|
| **Feature Extraction** | The backbone is frozen, and only the fully connected (FC) layer is trained. | Small datasets, similar tasks |
| **Fine-Tuning** | The entire model is trained using a low learning rate. | Sufficient data, different tasks |

> **💡 Tip:** During **fine-tuning**, keep the learning rate low (e.g., **1e-4** or **1e-5**) to avoid disrupting the pre-trained weights.

## Training Cycle (Trainer)

In [ ]:
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from tqdm import tqdm

def train_epoch(model, dataloader, optimizer, criterion, device, scaler=None):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in tqdm(dataloader, desc='Training'):
        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if scaler:
          with autocast():
            outputs = model(inputs)
            loss = criterion(outputs, labels)
          scaler.scale(loss).backward()
          scaler.unscale_(optimizer)
          torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
          scaler.step(optimizer)
          scaler.update()
        else:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return running_loss / total, 100. * correct / total

def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.inference_mode():
        for inputs, labels in tqdm(dataloader, desc='Evaluating'):
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return running_loss / total, 100. * correct / total

### Train Tuning

In [ ]:
def train_model(model, trainloader, valloader, epochs=50, lr=1e-3, device='cuda'):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = GradScaler()

    best_val_asc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, trainloader, optimizer, criterion, device, scaler)
        val_loss, val_acc = evaluate(model, valloader, criterion, device)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f'Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f} - Train Acc: {train_acc:.2f}% - Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.2f}%')


        if val_acc > best_val_asc:
            best_val_asc = val_acc
            torch.save(model.state_dict(), 'best_model.pth')

    return model, history

> **⚠️ Important:** Use `set_to_none=True` when calling `optimizer.zero_grad()` for faster gradient resetting **[5, pp. 58–60]**.
Prefer `torch.inference_mode()` over `torch.no_grad()` during inference for better evaluation performance.

## Comparative Training

In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

#1. CNN from scratch
print("\n" + "="*50)
print("Training CNN from Scratch")
print("="*50)
cnn_model = CNNFromScratch(num_classes=10)
cnn_model, cnn_history = train_model(cnn_model, trainloader, valloader, epochs=50, lr=1e-3, device=device)

#2. Fine-Tuning ResNet-18
print("\n" + "="*50)
print("Training ResNet-18 with Fine-Tuning")
print("="*50)
resnet_model = get_resnet18(num_classes=10, freeze_backbone=False)
resnet_model, resnet_history = train_model(resnet_model, trainloader, valloader, epochs=50, lr=1e-4, device=device)

#3. Feature Extractor ResNet-18
print("\n" + "="*50)
print("Training ResNet-18 as Feature Extractor")
print("="*50)
resnet_fe_model = get_resnet18(num_classes=10, freeze_backbone=True)
resnet_fe_model, resnet_fe_history = train_model(resnet_fe_model, trainloader, valloader, epochs=50, lr=1e-3, device=device)



## Visualizing Results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_history(histories, labels):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss
    for hist, label in zip(histories, labels):
        axes[0].plot(hist['train_loss'], '--', label=f'{label} Train')
        axes[0].plot(hist['val_loss'], '-', label=f'{label} Val')
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()

    # Accuracy
    for hist, label in zip(histories, labels):
        axes[1].plot(hist['train_acc'], '--', label=f'{label} Train')
        axes[1].plot(hist['val_acc'], '-', label=f'{label} Val')
    axes[1].set_title('Accuracy (%)')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('training_comparison.png', dpi=150)
    plt.show()

histories = [cnn_history, resnet_history, resnet_fe_history]
labels = ['CNN Scratch', 'ResNet FT', 'ResNet FE']
plot_history(histories, labels)

### Expected Results (CIFAR-10)

| Model | Test Accuracy | Training Time | Parameters |
|-------|--------------:|--------------:|-----------:|
| CNN (Trained from Scratch) | ~75–80% | ~45 min | ~391K |
| ResNet-18 (Fine-Tuned) | ~92–94% | ~60 min | ~11.2M |
| ResNet-18 (Feature Extraction) | ~85–88% | ~30 min | ~11.2M |

## Test Set Evaluation

In [ ]:
# Load and test the best model
model = CNNFromScratch(num_classes=10)
model.load_state_dict(torch.load('best_model.pth'))
model.to(device)

test_loss, test_acc = evaluate(model, testloader, nn.CrossEntropyLoss(), device)
print(f"Test Accuracy: {test_acc:.2f}%")

#ConfusionMatrix
from sklearn.metrics import confusion_matrix
import numpy as np

def plot_confusion_matrix(model, dataloader, device, class_names):
  model.eval()
  all_preds = []
  all_labels = []

  with torch.inference_mode():
    for inputs, labels in dataloader:
      inputs, labels = inputs.to(device), labels.to(device)
      outputs = model(inputs)
      _, preds = outputs.max(1)
      all_preds.extend(preds.cpu().numpy())
      all_labels.extend(labels.cpu().numpy())

  cm = confusion_matrix(all_labels, all_preds)
  plt.figure(figsize=(10, 8))
  sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
  plt.xlabel('Predicted')
  plt.ylabel('True')
  plt.title('Confusion Matrix')
  plt.tight_layout()
  plt.savefig('confusion_matrix.png', dpi=150)
  plt.show()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
plot_confusion_matrix(model, testloader, device, class_names)

## Reproducibility Tuning

In [ ]:
import random
import numpy as np

def set_seed(seed=42):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)

  # CuDNN deterministic mode
  torch.backends.cudnn.deterministic = True
  torch.backends.cudnn.benchmark = False
  torch.use_deterministic_algorithms(True, warn_only=True)

  # Environment variable for CUDA 10.2+ (in terminal)
  # export CUBLAS_WORKSPACE_CONFIG=:4096:8

set_seed(42)

> **⚠️ Important:** Reproducibility is a fundamental requirement in scientific research **[5, pp. 106–123]**. The settings described above are compatible with **PyTorch 2.x**.

# 📌 To-Do

- [ ] Compare the model with **EfficientNet-B0**
  ```python
  import timm

  model = timm.create_model(
      "efficientnet_b0",
      pretrained=True,
      num_classes=10
  )
  ```
  
- [ ] Visualize model attention using Grad-CAM
- [ ] Perform Hyperparameter Tuning using Optuna or Ray Tune
 - [ ] Add Distributed Data Parallel (DDP) support for multi-GPU training
 - [ ] Export the trained model to ONNX for production deploymen